# CNN Product Image Classifier
**CPSC 393 – Fall 2026** · Luke Wiljanen, Ashlee Nguyen & Auggie Pirazzi

We compare four models on Kaggle's [Fashion Product Images (Small)](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small) to see where classification performance comes from: a fully connected network, a CNN from scratch, a convolutional autoencoder, and a fine-tuned ResNet50. All four use the same split, callbacks, and evaluation code, so the model is the only thing that changes.

| Section | Owner | Status |
|---|---|---|
| 1. Setup | All | ✅ |
| 2. Data check | Auggie / Luke | ✅ |
| 3. Splits | Luke | ✅ |
| 4. Loaders, training, evaluation | Ashlee | ⬜ |
| 5. Fully connected baseline | Luke | ⬜ |
| 6. CNN from scratch | Ashlee | ⬜ |
| 7. ResNet50 | Auggie | ⬜ |
| 8. Autoencoder | Luke | ⬜ |
| 9. Final test evaluation and figures | All | ⬜ |

**Working in one notebook as a team:** only edit your own section, and run *Kernel → Restart & Clear Output* before committing. Notebook outputs cause most merge conflicts.

## 1. Setup

In [ ]:
import csv
import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

try:
    import tensorflow as tf
    print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
except ImportError:
    print("TensorFlow not installed yet (only needed from section 4 on).")

In [ ]:
# ---- shared settings: change these here only ----
SEED = 42
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))   # Kaggle download lands here (git-ignored)
SPLITS_DIR = Path("splits")                            # committed, so everyone uses the same split
FIGURES_DIR = Path("figures")

# articleType values exactly as they appear in styles.csv
CLASSES_7 = ["Tshirts", "Casual Shoes", "Watches", "Kurtas", "Handbags", "Heels", "Sunglasses"]
CLASSES_EXTRA = ["Shirts", "Tops", "Sports Shoes"]     # added for the ten-class run
CLASSES_10 = CLASSES_7 + CLASSES_EXTRA

PER_CLASS = 1000
SPLIT_SIZES = {"train": 700, "val": 150, "test": 150}  # 70 / 15 / 15

IMG_H, IMG_W = 80, 60       # small-dataset images are 60 wide x 80 tall
RESNET_H, RESNET_W = 128, 96

for d in (DATA_DIR, SPLITS_DIR, FIGURES_DIR):
    d.mkdir(exist_ok=True)

## 2. Data check
Downloads the dataset the first time (about 600 MB) with `kagglehub`. If that fails, download the zip from Kaggle and unzip it into `data/`.

In [ ]:
def find_dataset(data_dir=DATA_DIR):
    hits = sorted(Path(data_dir).rglob("styles.csv"))
    if not hits:
        return None
    styles = hits[0]
    images = styles.parent / "images"
    return (styles, images) if images.is_dir() else None

found = find_dataset()
if found is None:
    import kagglehub   # pip install kagglehub
    path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")
    found = find_dataset(path)
STYLES_CSV, IMAGES_DIR = found
print(STYLES_CSV, "|", IMAGES_DIR)

In [ ]:
COLUMNS = ["id", "gender", "masterCategory", "subCategory", "articleType",
           "baseColour", "season", "year", "usage", "productDisplayName"]

def load_styles(path):
    """Read styles.csv. A few product names contain commas, which gives those rows
    extra fields and makes pd.read_csv fail. We join the extras back into the name
    so those products are kept rather than skipped."""
    rows, fixed = [], 0
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        assert next(reader) == COLUMNS, "unexpected header"
        for r in reader:
            if not r:
                continue
            if len(r) > len(COLUMNS):
                r, fixed = r[:9] + [",".join(r[9:])], fixed + 1
            rows.append(r + [""] * (len(COLUMNS) - len(r)))
    df = pd.DataFrame(rows, columns=COLUMNS).replace("", pd.NA)
    df["id"] = df["id"].astype(int)
    return df, fixed

def image_path(pid):
    return IMAGES_DIR / f"{pid}.jpg"

def load_image(pid, size=(IMG_W, IMG_H)):
    """RGB uint8 array of shape (H, W, 3). Converts grayscale/RGBA and resizes odd sizes."""
    with Image.open(image_path(pid)) as im:
        im = im.convert("RGB")
        if im.size != size:
            im = im.resize(size, Image.BILINEAR)
        return np.asarray(im)

styles, n_fixed = load_styles(STYLES_CSV)
on_disk = {int(p.stem) for p in IMAGES_DIR.glob("*.jpg") if p.stem.isdigit()}
styles["has_image"] = styles["id"].isin(on_disk)

print(f"rows: {len(styles):,}  (rows with commas in the name, kept: {n_fixed})")
print(f"article types: {styles['articleType'].nunique()}")
print(f"missing articleType: {styles['articleType'].isna().sum()}")
print(f"products with no image file: {(~styles['has_image']).sum()}")
print("\nmissing values by column:")
print(styles[COLUMNS].isna().sum()[lambda s: s > 0].to_string())

In [ ]:
have = styles[styles["has_image"]]
counts = (have["articleType"].value_counts()
          .reindex(CLASSES_10).rename("products with image").to_frame())
counts["run"] = ["7 and 10" if c in CLASSES_7 else "10 only" for c in counts.index]
counts

In [ ]:
# image sizes and color modes, so we know how many the loader has to fix
shapes = Counter()
for pid in have["id"]:
    with Image.open(image_path(pid)) as im:
        shapes[(im.size, im.mode)] += 1
size_table = pd.DataFrame([(f"{w}x{h}", m, n) for ((w, h), m), n in shapes.most_common()],
                          columns=["size (WxH)", "mode", "images"])
print(f"not {IMG_W}x{IMG_H} RGB:", sum(n for (s, m), n in shapes.items() if s != (IMG_W, IMG_H) or m != "RGB"))
size_table

In [ ]:
# sample grid: one row per class (goes in the slides)
rng = np.random.default_rng(SEED)
k = 8
fig, axes = plt.subplots(len(CLASSES_7), k, figsize=(k * 1.1, len(CLASSES_7) * 1.45))
for r, c in enumerate(CLASSES_7):
    ids = rng.choice(have.loc[have["articleType"] == c, "id"].to_numpy(), size=k, replace=False)
    for j, pid in enumerate(ids):
        ax = axes[r, j]
        ax.imshow(load_image(pid))
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(c, rotation=0, ha="right", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "sample_grid.png", dpi=150)
plt.show()

## 3. Splits
Built once with a fixed seed and saved to `splits/`. If the CSVs already exist, this section loads them instead of re-splitting, so every model and every teammate trains on the same images.

- **Seven-class:** 1,000 images per class → 4,900 train / 1,050 val / 1,050 test.
- **Ten-class:** the same seven-class split plus 1,000 each of shirts, tops, and sports shoes (700 / 150 / 150 each).
- **Unlabeled pool** (autoencoder): every product with an image except the test images of *both* runs, so no test image influences feature learning.

In [ ]:
SPLIT_FILES = {"seven": SPLITS_DIR / "split_7class.csv",
               "ten": SPLITS_DIR / "split_10class.csv",
               "pool": SPLITS_DIR / "unlabeled_pool.csv"}

def sample_class(df, cls, rng):
    ids = np.sort(df.loc[df["articleType"] == cls, "id"].to_numpy())
    if len(ids) < PER_CLASS:
        raise ValueError(f"{cls} has only {len(ids)} images with files")
    ids = rng.permutation(ids)[:PER_CLASS]
    parts, start = [], 0
    for name, n in SPLIT_SIZES.items():
        parts.append(pd.DataFrame({"id": ids[start:start + n], "label": cls, "split": name}))
        start += n
    return pd.concat(parts)

def build_splits(have):
    rng = np.random.default_rng(SEED)
    seven = pd.concat([sample_class(have, c, rng) for c in CLASSES_7], ignore_index=True)
    extra = pd.concat([sample_class(have, c, rng) for c in CLASSES_EXTRA], ignore_index=True)
    ten = pd.concat([seven, extra], ignore_index=True)
    test_ids = set(ten.loc[ten["split"] == "test", "id"])
    pool = have.loc[~have["id"].isin(test_ids), ["id"]].sort_values("id").reset_index(drop=True)
    return seven, ten, pool

if all(p.exists() for p in SPLIT_FILES.values()):
    seven, ten, pool = (pd.read_csv(SPLIT_FILES[k]) for k in ("seven", "ten", "pool"))
    print("Loaded existing splits from", SPLITS_DIR)
else:
    seven, ten, pool = build_splits(have)
    for k, df in zip(("seven", "ten", "pool"), (seven, ten, pool)):
        df.to_csv(SPLIT_FILES[k], index=False)
    print("Built and saved new splits to", SPLITS_DIR, "- commit these files.")

In [ ]:
# sanity checks: sizes, balance, and no overlap between train/val/test or with the pool
for name, df, classes in (("seven", seven, CLASSES_7), ("ten", ten, CLASSES_10)):
    table = pd.crosstab(df["label"], df["split"]).reindex(index=classes, columns=list(SPLIT_SIZES))
    assert (table == pd.Series(SPLIT_SIZES)).all().all(), f"{name}: unbalanced"
    assert df["id"].is_unique, f"{name}: an image appears in two splits"
    print(f"{name}-class:", df["split"].value_counts().reindex(list(SPLIT_SIZES)).to_dict())

assert set(seven["id"]) <= set(ten["id"]), "seven-class split should be inside the ten-class split"
test_ids = set(ten.loc[ten["split"] == "test", "id"])
assert not test_ids & set(pool["id"]), "a test image leaked into the unlabeled pool"
print(f"unlabeled pool: {len(pool):,} images (all products with an image minus {len(test_ids):,} test images)")

## 4. Loaders, training, and evaluation (Ashlee)
Shared by every model:
- `make_dataset(split_df, split, size, augment)` → `tf.data.Dataset`, normalized with training-split channel stats; augmentation (horizontal flip, small rotation, mild color jitter) on the training split only.
- `train(model, ...)` → Adam, batch 64, EarlyStopping + ReduceLROnPlateau on `val_loss`; saves the history.
- `evaluate(model, ...)` → accuracy, macro F1, confusion matrix, and coverage/accuracy at 90% confidence.

In [ ]:
# TODO (Ashlee)

## 5. Fully connected baseline (Luke)
Flatten (14,400) → 512 → 128 → output, ReLU and dropout, lr 1e-3. `model.summary()` should show 7,439,879 parameters.

In [ ]:
# TODO (Luke)

## 6. CNN from scratch (Ashlee)
Three blocks of 3×3 conv (32, 64, 128) + batch norm + ReLU + 2×2 max pool → global average pooling → output. Expect 94,599 **trainable** parameters (95,047 total, since batch norm's moving mean and variance aren't trained).

In [ ]:
# TODO (Ashlee)

## 7. ResNet50 (Auggie)
`ResNet50(weights="imagenet", include_top=False)` → global average pooling → output. Inputs resized to 128×96 and passed through `tf.keras.applications.resnet50.preprocess_input`. lr 1e-4.

In [ ]:
# TODO (Auggie)

## 8. Autoencoder (Luke)
Three stride-2 convs → 128-d code → mirrored decoder, MSE loss on the unlabeled pool. **Nov 20 checkpoint:** keep it only if the reconstructions are recognizable. Then freeze the encoder and train a small classifier on the codes.

In [ ]:
# TODO (Luke)

## 9. Final test evaluation and figures (All)
Run once, after all tuning is finished: results table, confusion matrices, loss curves, misclassified examples, feature maps, Grad-CAM, and the autoencoder figures.

In [ ]:
# TODO (All)